# ⚖️ SQL Quality Judge

**Workshop hands-on:** postavíme automatickou evaluaci AI agenta,
který generuje SQL z přirozeného jazyka.

## Co budeme dělat

1. 🗄️ Vytvoříme malou **e-shop SQLite databázi**
2. 🤖 Postavíme **SQL agenta** (LLM + schema v promptu)
3. ⚖️ Postavíme **soudce** (LLM-as-judge)
4. 📏 Spustíme **3 úrovně evaluace** na 12 test cases

Žádný MCP, žádný cloud DB. Vše lokálně + 1 LLM přes OpenRouter.

In [ ]:
import subprocess
import sys

try:
    import openrouter
except ImportError:
    print("Installing openrouter...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "openrouter"]
    )
    import openrouter
print(f"✓ openrouter ready")

## 🔑 OpenRouter API klíč

V **Google Colab** klikni vlevo na klíček 🔑 a přidej secret
`OPENROUTER_API_KEY` s hodnotou tvého klíče z [openrouter.ai/keys](https://openrouter.ai/keys).

Mimo Colab nastav environment variable:
```bash
export OPENROUTER_API_KEY=sk-or-...
```

In [ ]:
import os

def get_api_key():
    # Try Google Colab secrets first
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except (ImportError, Exception):
        pass
    # Fallback to env var
    return os.environ.get("OPENROUTER_API_KEY")

api_key = get_api_key()
if not api_key:
    raise RuntimeError(
        "OPENROUTER_API_KEY nenalezen. V Colab použij Secrets 🔑, "
        "lokálně nastav environment variable."
    )
print("✓ API key načten")

---

## 📚 Krátká teorie

**Eval = automatický test kvality AI výstupu.**

| | Unit test | Eval |
|---|---|---|
| Cíl | správnost | kvalita |
| Výstup | PASS / FAIL | skóre |
| Determinismus | stejný vstup = stejný výstup | různý výstup každý run |
| Co testuje | funkci | model + prompt + data |

Test ti řekne *"kód funguje"*. Eval ti řekne *"agent dělá dobrou práci"*.

### Z čeho se eval skládá

- 🗂️ **Test dataset** — sada otázek + správná odpověď
- 🤖 **Generator** — tvůj agent (otázka → SQL)
- 📏 **Metrika** — jak měřit shodu
- 📊 **Agregace** — souhrn napříč případy

100 otázek + agent + metrika → *"87 % správně"*.

---

## 🗄️ E-shop databáze

4 tabulky · 55 řádků · 99 420 Kč v transakcích.

In [ ]:
import sqlite3

SCHEMA = """
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    email         TEXT NOT NULL,
    city          TEXT,
    country       TEXT,
    signup_date   TEXT
);
CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    category      TEXT,
    price         REAL,
    stock_qty     INTEGER
);
CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER NOT NULL,
    order_date    TEXT,
    status        TEXT,
    total_amount  REAL
);
CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id      INTEGER NOT NULL,
    product_id    INTEGER NOT NULL,
    quantity      INTEGER NOT NULL,
    unit_price    REAL
);
"""

CUSTOMERS = [
    (1,  "Anna Nováková",       "anna.novakova@email.cz",  "Praha",      "CZ", "2024-01-15"),
    (2,  "Petr Svoboda",        "petr.svoboda@email.com",  "Brno",       "CZ", "2024-02-20"),
    (3,  "Markéta Procházková", "mp@mail.cz",              "Ostrava",    "CZ", "2024-03-10"),
    (4,  "Tomáš Dvořák",        "tomas.d@gmail.com",       "Praha",      "CZ", "2024-04-05"),
    (5,  "Eva Černá",           "eva.cerna@email.cz",      "Plzeň",      "CZ", "2024-05-12"),
    (6,  "Jan Horák",           "jan.horak@example.com",   "Bratislava", "SK", "2024-06-18"),
    (7,  "Hana Veselá",         "h.vesela@mail.cz",        "Praha",      "CZ", "2024-07-22"),
    (8,  "Michael Krejčí",      "mkrejci@gmail.com",       "Vienna",     "AT", "2024-08-30"),
    (9,  "Lucie Marešová",      "lucie.maresova@email.cz", "Brno",       "CZ", "2024-09-14"),
    (10, "David Pokorný",       "d.pokorny@example.com",   "Praha",      "CZ", "2024-10-08"),
]

PRODUCTS = [
    (1,  "Laptop Lenovo X1",       "Electronics", 32000.00, 5),
    (2,  "Kniha 1984",             "Books",         350.00, 50),
    (3,  "Tričko bílé",            "Clothing",      450.00, 30),
    (4,  "Hrnek modrý",            "Home",          250.00, 100),
    (5,  "Sluchátka Sony WH-1000", "Electronics",  3500.00, 15),
    (6,  "Kniha Hobit",            "Books",         420.00, 25),
    (7,  "Mikina šedá",            "Clothing",     1200.00, 20),
    (8,  "Polštář dekorační",      "Home",          800.00, 40),
    (9,  "Tablet Samsung Tab A",   "Electronics",  8500.00, 8),
    (10, "Kniha Solaris",          "Books",         380.00, 30),
]

ORDERS = [
    (1,  1,  "2024-03-01", "completed", 32000.00),
    (2,  2,  "2024-03-15", "completed",  1200.00),
    (3,  3,  "2024-04-02", "completed",   700.00),
    (4,  1,  "2024-04-20", "completed",  8500.00),
    (5,  4,  "2024-05-10", "cancelled",   420.00),
    (6,  5,  "2024-05-25", "completed",  3750.00),
    (7,  2,  "2024-06-12", "completed",  3500.00),
    (8,  6,  "2024-06-30", "completed",  1250.00),
    (9,  7,  "2024-07-15", "completed",   800.00),
    (10, 8,  "2024-08-05", "completed",  8500.00),
    (11, 3,  "2024-08-22", "pending",     350.00),
    (12, 9,  "2024-09-10", "completed",  1580.00),
    (13, 1,  "2024-09-28", "completed",   920.00),
    (14, 10, "2024-10-15", "completed", 32000.00),
    (15, 5,  "2024-11-03", "pending",    3950.00),
]

ORDER_ITEMS = [
    (1,   1,  1, 1, 32000.00), (2,   2,  7, 1,  1200.00),
    (3,   3,  2, 2,   350.00), (4,   4,  9, 1,  8500.00),
    (5,   5,  6, 1,   420.00), (6,   6,  5, 1,  3500.00),
    (7,   6,  4, 1,   250.00), (8,   7,  5, 1,  3500.00),
    (9,   8,  3, 1,   450.00), (10,  8,  8, 1,   800.00),
    (11,  9,  8, 1,   800.00), (12, 10,  9, 1,  8500.00),
    (13, 11,  2, 1,   350.00), (14, 12,  7, 1,  1200.00),
    (15, 12, 10, 1,   380.00), (16, 13,  4, 2,   250.00),
    (17, 13,  6, 1,   420.00), (18, 14,  1, 1, 32000.00),
    (19, 15,  5, 1,  3500.00), (20, 15,  3, 1,   450.00),
]

# In-memory SQLite — žádný soubor, žádný cleanup
conn = sqlite3.connect(":memory:")
conn.executescript(SCHEMA)
conn.executemany("INSERT INTO customers   VALUES (?, ?, ?, ?, ?, ?)", CUSTOMERS)
conn.executemany("INSERT INTO products    VALUES (?, ?, ?, ?, ?)",    PRODUCTS)
conn.executemany("INSERT INTO orders      VALUES (?, ?, ?, ?, ?)",    ORDERS)
conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",    ORDER_ITEMS)
conn.commit()

print(f"✓ DB ready · {len(CUSTOMERS)} customers · {len(PRODUCTS)} products · "
      f"{len(ORDERS)} orders · {len(ORDER_ITEMS)} order_items")

### 👀 Mrkněme, co je uvnitř

In [ ]:
# Top 5 zákazníků a kolik utratili (jen completed orders)
rows = conn.execute(
    """
    SELECT c.name, ROUND(SUM(o.total_amount), 0) AS spent
    FROM customers c JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id
    ORDER BY spent DESC
    LIMIT 5
    """
).fetchall()
for name, spent in rows:
    print(f"  {name:25s} {spent:>8.0f} Kč")

---

## 🤖 SQLAgent

Agent = LLM se schematem v system promptu. Dostane otázku, vrátí SQL.
Žádný MCP, žádné nástroje — jen prompt + structured JSON output.

In [ ]:
import json
from openrouter import OpenRouter

DEFAULT_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

AGENT_SYSTEM_PROMPT = f"""Jsi SQL agent pro SQLite e-shop databázi.

DATABÁZE SCHEMA:
{SCHEMA}

Pravidla:
- Vrať JSON s jediným polem "sql" obsahujícím čistý SQL query.
- SQLite syntax (pro datum: strftime, LIKE '2024%').
- Tabulky a sloupce použij přesně jak jsou ve schéma (lowercase).
"""

AGENT_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "sql_query",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "sql": {"type": "string"},
            },
            "required": ["sql"],
            "additionalProperties": False,
        },
    },
}

def generate_sql(question: str, model: str = DEFAULT_MODEL) -> str:
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model=model,
            messages=[
                {"role": "system", "content": AGENT_SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            response_format=AGENT_RESPONSE_FORMAT,
        )
    return json.loads(response.choices[0].message.content)["sql"].strip()

### 🎯 Vyzkoušej agenta

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

question_input = widgets.Text(
    value="Kolik zákazníků je z Prahy?",
    placeholder="Zeptej se na něco z e-shop databáze...",
    description="Otázka:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "initial"},
)

run_agent_button = widgets.Button(
    description="🤖 Generuj SQL",
    button_style="primary",
)

output = widgets.Output()


def on_click(_):
    with output:
        clear_output()
        display(Markdown("⏳ *Generuji SQL... (volání LLM trvá ~3 s)*"))
    try:
        sql = generate_sql(question_input.value)
        with output:
            clear_output()
            display(Markdown(f"**Vygenerovaný SQL:**\n\n```sql\n{sql}\n```"))
            try:
                rows = conn.execute(sql).fetchall()
                result_str = "\n".join(f"  {row}" for row in rows[:10])
                display(Markdown(f"**Výsledek:**\n\n```\n{result_str}\n```"))
            except Exception as e:
                display(Markdown(f"❌ **Chyba:** `{e}`"))
    except Exception as e:
        with output:
            clear_output()
            display(Markdown(f"❌ **LLM call failed:** `{e}`"))


run_agent_button.on_click(on_click)
display(widgets.VBox([question_input, run_agent_button, output]))

---

## ⚖️ SQLJudge

Judge = LLM, který se podívá na otázku + vygenerovaný SQL + referenční SQL
a vrátí **skóre 0–10 + verdict PASS/FAIL + seznam issues**.

Stejný structured JSON output, jen jiný prompt.

In [ ]:
from openrouter import OpenRouter as ORClient

JUDGE_PROMPT = f"""Posuď, jestli vygenerovaný SQL odpovídá na otázku.

DATABÁZE SCHEMA:
{SCHEMA}

OTÁZKA UŽIVATELE:
{{question}}

VYGENEROVANÝ SQL:
{{generated}}

REFERENČNÍ SQL (správné řešení):
{{expected}}

VÝSLEDEK GENERATED SQL:
{{result}}

Kritéria:
1. Sémantika — odpovídá to na otázku?
2. Edge cases — NULL handling, status filter, ORDER BY
3. Smysluplnost — žádné latentní bugy

Skórování: 10 = perfektní · 7-9 = funguje · 4-6 = částečně · 0-3 = špatně.
PASS pokud score >= 7.
"""

JUDGE_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "sql_judgment",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "score":   {"type": "integer", "minimum": 0, "maximum": 10},
                "verdict": {"type": "string", "enum": ["PASS", "FAIL"]},
                "issues":  {"type": "array", "items": {"type": "string"}},
            },
            "required": ["score", "verdict", "issues"],
            "additionalProperties": False,
        },
    },
}

def judge_sql(question, generated, expected, result="") -> dict:
    prompt = JUDGE_PROMPT.format(
        question=question,
        generated=generated,
        expected=expected,
        result=str(result)[:500] or "(nespuštěno)",
    )
    with ORClient(api_key=api_key) as client:
        response = client.chat.send(
            model=DEFAULT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format=JUDGE_RESPONSE_FORMAT,
        )
    return json.loads(response.choices[0].message.content)

---

## 🗂️ Test cases

12 dvojic `(otázka, referenční SQL)`. Reference SQLs jsme napsali ručně
a ověřili, že vracejí správné výsledky.

In [ ]:
TEST_CASES = [
    ("Kolik máme zákazníků celkem?",
     "SELECT COUNT(*) FROM customers"),
    ("Kolik zákazníků je z Prahy?",
     "SELECT COUNT(*) FROM customers WHERE city = 'Praha'"),
    ("Top 3 produkty podle prodaných kusů",
     """SELECT p.name, SUM(oi.quantity) AS qty
        FROM order_items oi JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.product_id ORDER BY qty DESC LIMIT 3"""),
    ("Celkové tržby v 2024 (jen dokončené objednávky)",
     """SELECT SUM(total_amount) FROM orders
        WHERE status = 'completed' AND order_date LIKE '2024%'"""),
    ("Která kategorie produktů má nejvyšší průměrnou cenu?",
     """SELECT category, AVG(price) AS avg_price
        FROM products GROUP BY category
        ORDER BY avg_price DESC LIMIT 1"""),
    ("Který zákazník udělal nejvíc objednávek?",
     """SELECT c.name, COUNT(*) AS n
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id ORDER BY n DESC LIMIT 1"""),
    ("Počet objednávek podle statusu",
     "SELECT status, COUNT(*) FROM orders GROUP BY status ORDER BY status"),
    ("Produkty skladem méně než 20 kusů",
     "SELECT name, stock_qty FROM products WHERE stock_qty < 20 ORDER BY stock_qty"),
    ("Která objednávka má nejvyšší hodnotu?",
     "SELECT order_id, total_amount FROM orders ORDER BY total_amount DESC LIMIT 1"),
    ("Kolik kusů knih se prodalo celkem?",
     """SELECT SUM(oi.quantity)
        FROM order_items oi JOIN products p ON oi.product_id = p.product_id
        WHERE p.category = 'Books'"""),
    ("Zákazníci, kteří nikdy nic neobjednali",
     """SELECT c.name FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_id IS NULL"""),
    ("Měsíční tržby v 2024 (jen completed)",
     """SELECT strftime('%Y-%m', order_date) AS month, SUM(total_amount) AS revenue
        FROM orders WHERE status = 'completed' AND order_date LIKE '2024%'
        GROUP BY month ORDER BY month"""),
]
print(f"✓ {len(TEST_CASES)} test cases ready")

---

## 📏 Tři úrovně evaluace

### Level 1 · String match — *nejjednodušší, nejhorší*
Porovná dvě SQL stringy (case + whitespace insensitive).
V reálu fail-uje pořád, protože LLM píše SQL trochu jinak pokaždé.

### Level 2 · Result match — *funkční ekvivalence*
Spustí oba SQL a porovná výsledky. Odolné vůči formátování,
ale nezachytí latentní bugy.

### Level 3 · LLM as judge — *sémantická kvalita*
LLM posoudí kvalitu výstupu na škále 0–10 + vrátí issues.
Drahé a nedeterministické, ale chápe sémantiku.

In [ ]:
def string_match(generated: str, expected: str) -> bool:
    """Level 1 — normalize whitespace + case, compare."""
    norm = lambda s: " ".join(s.lower().split())
    return norm(generated) == norm(expected)

def run_and_sort(sql: str, conn) -> list:
    cur = conn.execute(sql)
    rows = cur.fetchall()
    # Sort by string repr — neutralizuje pořadí řádků
    return sorted([tuple(r) for r in rows], key=str)

def result_match(generated: str, expected: str, conn) -> tuple[bool, str]:
    """Level 2 — spustí oba a porovná výsledky."""
    try:
        gen_result = run_and_sort(generated, conn)
        exp_result = run_and_sort(expected, conn)
        return gen_result == exp_result, str(gen_result)[:200]
    except Exception as e:
        return False, f"ERROR: {e}"

print("✓ metric functions ready")

### 🎯 Vyzkoušej všechny 3 úrovně na jednom test case

Než pustíme celou evaluaci, podíváme se, jak každá úroveň funguje **izolovaně**.
               
Vezmeme jednu otázku, agent vygeneruje SQL a aplikujeme všechny 3 metriky postupně.

In [ ]:
# Vyber jeden test case
question, expected_sql = TEST_CASES[5]

print(f"📋 Otázka:        {question}")
print(f"📋 Reference SQL: {' '.join(expected_sql.split())}")
print()

print("⏳ Agent generuje SQL...")
generated_sql = generate_sql(question)
print(f"✓ Generated SQL: {generated_sql}")
print()
print("=" * 60)

# ─── Level 1: String match ─────────────────────────────────
print("\n── Level 1 · String match ──")
l1 = string_match(generated_sql, expected_sql)
print(f"   {'✅ PASS' if l1 else '❌ FAIL'}   — porovnává SQL jako text")

# ─── Level 2: Result match ─────────────────────────────────
print("\n── Level 2 · Result match ──")
l2, gen_result = result_match(generated_sql, expected_sql, conn)
print(f"   Generated výsledek: {gen_result[:100]}")
print(f"   {'✅ PASS' if l2 else '❌ FAIL'}   — spustí oba SQL a porovná data")

# ─── Level 3: LLM as judge ─────────────────────────────────
print("\n── Level 3 · LLM as judge ──")
print("   ⏳ Judge hodnotí... (~3 s)")
l3 = judge_sql(question, generated_sql, expected_sql, gen_result)
print(f"   Score:   {l3['score']}/10")
print(f"   Verdict: {l3['verdict']}")
if l3.get("issues"):
    print(f"   Issues:")
    for issue in l3["issues"]:
        print(f"     └─ {issue}")

---

## 🚀 Spusť celý eval

Pro každý test case:
1. Agent vygeneruje SQL
2. Level 1: string match
3. Level 2: result match (proti SQLite)
4. Level 3: LLM-as-judge

Volá LLM dvakrát pro case (agent + judge).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

run_eval_button = widgets.Button(
    description="🚀 Spusť eval",
    button_style="success",
)

progress = widgets.IntProgress(
    value=0, min=0, max=len(TEST_CASES),
    description="0 / 0",
    layout=widgets.Layout(width="60%"),
)
progress.layout.visibility = "hidden"

eval_output = widgets.Output()
eval_results = []


def on_run_eval(_):
    global eval_results
    eval_results = []
    progress.max = len(TEST_CASES)
    progress.value = 0
    progress.description = f"0 / {len(TEST_CASES)}"
    progress.layout.visibility = "visible"

    with eval_output:
        clear_output()
        n = len(TEST_CASES)
        print(f"⚖️  SQL Quality Judge — {n} test cases\n")

    for i, (question, expected_sql) in enumerate(TEST_CASES, 1):
        with eval_output:
            print(f"[{i:2d}/{n}] {question}")

        generated_sql = generate_sql(question)
        l1 = string_match(generated_sql, expected_sql)
        l2, gen_result = result_match(generated_sql, expected_sql, conn)
        l3 = judge_sql(question, generated_sql, expected_sql, gen_result)

        with eval_output:
            l1_i = "✅" if l1 else "❌"
            l2_i = "✅" if l2 else "❌"
            l3_i = "✅" if l3.get("verdict") == "PASS" else "❌"
            print(
                f"        L1 string {l1_i}  L2 result {l2_i}  "
                f"L3 judge {l3_i} {l3.get('score', 0)}/10"
            )
            if not l1:
                print(f"            generated: {' '.join(generated_sql.split())}")
                print(f"            expected:  {' '.join(expected_sql.split())}")
            for issue in l3.get("issues", []):
                print(f"            └─ {issue}")

        eval_results.append({
            "question": question,
            "generated": generated_sql,
            "l1": l1,
            "l2": l2,
            "l3_score": l3.get("score", 0),
            "l3_verdict": l3.get("verdict", "FAIL"),
        })

        progress.value = i
        progress.description = f"{i} / {n}"

    # Final summary
    with eval_output:
        l1_pass = sum(r["l1"] for r in eval_results)
        l2_pass = sum(r["l2"] for r in eval_results)
        l3_pass = sum(r["l3_verdict"] == "PASS" for r in eval_results)
        l3_avg = sum(r["l3_score"] for r in eval_results) / n

        bar = "═" * 47
        print(f"\n{bar}")
        print(f"  Level 1 (string match):   {l1_pass:>2d} / {n}  ({l1_pass * 100 // n:>3d} %)")
        print(f"  Level 2 (result match):   {l2_pass:>2d} / {n}  ({l2_pass * 100 // n:>3d} %)")
        print(f"  Level 3 (LLM as judge):   {l3_pass:>2d} / {n}  PASS · avg {l3_avg:.1f}/10")
        print(bar)


run_eval_button.on_click(on_run_eval)
display(widgets.VBox([run_eval_button, progress, eval_output]))